# Notebook 1 — prédiction du score d'examen

Projet ML 2026.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, learning_curve, KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from src import config, data, preprocessing, models, explain, features
from src.models import train_mlp, mlp_predict, regression_metrics, metrics_to_df
warnings.filterwarnings('ignore')
sns.set_theme(context='notebook', style='whitegrid')
plt.rcParams['figure.dpi'] = 110
config.FIG_DIR.mkdir(parents=True, exist_ok=True)
config.RES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = config.RANDOM_STATE


## Données

In [ ]:
df_full = data.load_train()
print('shape:', df_full.shape)
df_full.head(5)


In [ ]:
info = pd.DataFrame({
    'dtype': df_full.dtypes.astype(str),
    'n_unique': df_full.nunique(),
    'missing_pct': (df_full.isna().mean() * 100).round(2),
})
info


630k lignes, 14 variables. 3 colonnes ont des NaN (accès_internet, méthode_etude, heures_etude). Pas d'outliers chelous.

## EDA

In [ ]:
# Q1 — La cible est-elle équilibrée ? Faut-il un seuil de risque calibré ?
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(df_full[config.TARGET], bins=40, kde=True, color='#1f6feb', ax=axes[0])
axes[0].axvline(50, color='#d62728', ls='--', label='seuil tutorat (50)')
axes[0].set_title("Distribution de score_examen"); axes[0].set_xlabel('note'); axes[0].legend()
df_at = (df_full[config.TARGET] < 50).value_counts(normalize=True).rename({True:'à risque', False:'OK'})
df_at.plot(kind='bar', color=['#2ca02c','#d62728'], ax=axes[1])
axes[1].set_title('Proportion étudiants à risque (<50)'); axes[1].set_ylabel('fréquence')
for i, v in enumerate(df_at.values):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center')
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'eda_target.png', dpi=160); plt.show()


In [ ]:
# Q2 — Quelles variables numériques sont liées à la note ? (corr de Pearson)
num_corr = df_full[config.NUM_COLS_ALL + [config.TARGET]].corr()[config.TARGET].drop(config.TARGET).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
colors = ['#888' if abs(v) < 0.05 else '#1f6feb' for v in num_corr.values]
ax.barh(num_corr.index, num_corr.values, color=colors)
ax.axvline(0, color='k', lw=0.6)
ax.set_title('Corrélation de Pearson avec score_examen'); ax.set_xlabel('r')
for y, v in enumerate(num_corr.values):
    ax.text(v + (0.01 if v > 0 else -0.01), y, f'{v:+.2f}', va='center',
            ha='left' if v > 0 else 'right', fontsize=9)
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'eda_corr_target.png', dpi=160); plt.show()
print('Corrélations :')
print(num_corr.round(3))


heures_etude corrèle fort (~0.76), assiduité ~0.36, sommeil ~0.17. age, taille, fête : rien.

In [ ]:
# Q3 — Les variables catégorielles séparent-elles bien les notes ?
df_plot = df_full.copy()
for c in ['qualité_sommeil','méthode_etude','évaluation_établissement',
          'genre','accès_internet','difficulté_examen']:
    df_plot[c] = df_plot[c].astype('string').fillna('missing')
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.flat, ['qualité_sommeil','méthode_etude','évaluation_établissement',
                                'genre','accès_internet','difficulté_examen']):
    order = list(df_plot.groupby(col)[config.TARGET].mean().sort_values().index)
    sns.boxplot(data=df_plot, x=col, y=config.TARGET, order=order, ax=ax, fliersize=1)
    ax.set_title(col); ax.tick_params(axis='x', rotation=20); ax.set_xlabel('')
fig.suptitle('Note vs variables catégorielles', y=1.02)
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'eda_cat_boxplots.png', dpi=160); plt.show()


qualité_sommeil, méthode_etude et évaluation_établissement font des écarts ~10 pts. genre fait ~3 pts.

In [ ]:
# Q4 — Les variables manquantes sont-elles informatives ?
missing_summary = []
for col in ['heures_etude','accès_internet','méthode_etude']:
    m = df_full[col].isna()
    missing_summary.append({
        'variable': col,
        'pct_missing': m.mean()*100,
        'mean_score_when_missing': df_full.loc[m, config.TARGET].mean(),
        'mean_score_when_present': df_full.loc[~m, config.TARGET].mean(),
    })
miss_df = pd.DataFrame(missing_summary).round(2)
miss_df


Les NaN ne portent pas vraiment d'info, écart < 1 pt. Imputation médiane suffit.

In [ ]:
# Q5 — Effet conjoint heures_etude × méthode_etude (LE plus gros moteur de la note)
df_q5 = df_full.dropna(subset=['méthode_etude','heures_etude']).sample(40_000, random_state=RANDOM_STATE)
df_q5['méthode_etude'] = df_q5['méthode_etude'].astype('string')
g = sns.FacetGrid(df_q5, col='méthode_etude', col_wrap=3, height=2.8, sharey=True)
g.map_dataframe(sns.regplot, x='heures_etude', y=config.TARGET, scatter_kws={'alpha':.10,'s':6},
                line_kws={'color':'#d62728'})
for ax in g.axes.flat:
    ax.set_xlim(0, 10); ax.set_ylim(0, 105)
g.fig.suptitle('Note vs heures_etude, par méthode_etude', y=1.03)
g.fig.tight_layout(); g.fig.savefig(config.FIG_DIR / 'eda_he_methode.png', dpi=160); plt.show()


Pente positive partout, mais le niveau diffère selon la méthode (coaching/mixed > self-study).

## Sélection des variables

Gardé : heures_etude, assiduité_classe, heures_sommeil, genre, diplôme, accès_internet, méthode_etude, qualité_sommeil, évaluation_établissement.

Viré : age, taille_etudiant, heures_fête (corrélation nulle).

## Préprocessing

Pipeline pour éviter toute fuite (imputation/scaling fit que sur le train).

In [ ]:
pre = preprocessing.build_preprocessor()
print(pre)


Médiane pour les num, 'missing' pour les cat. OneHot + Standard. Tout dans un Pipeline.

## Split + comparaison des modèles

Full dataset (630k). Stratifié 70/15/15.

In [ ]:
df = data.load_train()  # 630_000 lignes
df = features.add_derived(df)
df = data.add_at_risk_label(df)
print('shape:', df.shape)
X_train, X_val, X_test, y_train, y_val, y_test = data.make_splits(df, features=config.FEATURES_FINAL_PLUS)
print('train', len(X_train), '| val', len(X_val), '| test', len(X_test))
print('at_risk : train', (y_train < 50).mean().round(3),
      '| val', (y_val < 50).mean().round(3),
      '| test', (y_test < 50).mean().round(3))


### CV 5-fold sur le train

In [ ]:
from sklearn.model_selection import cross_val_score
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_metrics = {}
cv_scores = {}
for name, pipe in models.make_sklearn_models().items():
    scores = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_scores[name] = scores
    pipe.fit(X_train, y_train)
    all_metrics[name] = models.regression_metrics(pipe, X_train, y_train, X_val, y_val, X_test, y_test)
    print(f'  {name:14s} CV RMSE={scores.mean():.3f}±{scores.std():.3f}  test_RMSE={all_metrics[name].rmse_test:.3f}')


In [ ]:
# 5.2 MLP PyTorch — architecture améliorée + scheduler cosine + early stopping
pre_fit = preprocessing.build_preprocessor()
Xtr = pre_fit.fit_transform(X_train).astype('float32')
Xva = pre_fit.transform(X_val).astype('float32')
Xte = pre_fit.transform(X_test).astype('float32')
print('Xtr shape:', Xtr.shape)
mlp, mlp_log = train_mlp(Xtr, y_train.to_numpy('float32'), Xva, y_val.to_numpy('float32'),
                          epochs=60, batch_size=2048, lr=3e-3,
                          hidden=(256,128,64), patience=8)
from src.models import RegMetrics
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score
pred_tr = mlp_predict(mlp, Xtr); pred_va = mlp_predict(mlp, Xva); pred_te = mlp_predict(mlp, Xte)
all_metrics['mlp_pytorch'] = RegMetrics(
    rmse_train = mean_squared_error(y_train, pred_tr) ** 0.5,
    rmse_val   = mean_squared_error(y_val, pred_va) ** 0.5,
    rmse_test  = mean_squared_error(y_test, pred_te) ** 0.5,
    mae_test   = mean_absolute_error(y_test, pred_te),
    r2_test    = r2_score(y_test, pred_te),
    acc_at_risk_test = accuracy_score(y_test < 50, pred_te < 50),
    f1_at_risk_test  = f1_score(y_test < 50, pred_te < 50),
)
import json
json.dump({k: v.tolist() for k, v in cv_scores.items()},
          open(config.RES_DIR / 'cv_scores.json','w'))
# Visualisation CV
fig, ax = plt.subplots(figsize=(7, 3.6))
names_cv = list(cv_scores.keys())
data_cv = [cv_scores[n] for n in names_cv]
ax.boxplot(data_cv, tick_labels=names_cv, showmeans=True)
ax.set_ylabel('RMSE (5-fold CV)'); ax.set_title('Distribution RMSE en 5-fold sur le train')
plt.xticks(rotation=15); fig.tight_layout()
fig.savefig(config.FIG_DIR / 'cv_boxplots.png', dpi=160); plt.show()


In [ ]:
table = metrics_to_df(all_metrics).sort_values('rmse_test')
table.to_csv(config.RES_DIR / 'failure_models_metrics.csv', index=False)
table


In [ ]:
# Comparaison RMSE train/val/test par modèle (pour over/underfitting)
tbl = table.set_index('model')[['rmse_train','rmse_val','rmse_test']]
fig, ax = plt.subplots(figsize=(8, 4))
tbl.plot(kind='bar', ax=ax, color=['#9ecae1','#3182bd','#08519c'])
ax.set_ylabel('RMSE'); ax.set_title('RMSE par split')
ax.tick_params(axis='x', rotation=15)
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'models_rmse_compare.png', dpi=160); plt.show()
# Courbe d'apprentissage MLP
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(mlp_log.train_rmse, label='train', color='#1f6feb')
ax.plot(mlp_log.val_rmse, label='val', color='#d62728')
ax.set_xlabel('epoch'); ax.set_ylabel('RMSE'); ax.set_title('MLP : train vs val par epoch')
ax.legend(); fig.tight_layout()
fig.savefig(config.FIG_DIR / 'mlp_curves.png', dpi=160); plt.show()


Les modèles non-linéaires battent Ridge et la baseline. Train ↔ val raisonnable, pas d'overfit grave.

## Hyperparamètres (HistGBRT)

RandomizedSearchCV, 3 folds, sur 30k pour aller vite.

In [ ]:
search_pipe = models.make_sklearn_models()['hist_gbrt']
param_dist = {
    'model__max_depth': [4, 6, 8, 10, None],
    'model__learning_rate': [0.02, 0.05, 0.08, 0.12],
    'model__max_iter': [200, 400, 600, 800],
    'model__min_samples_leaf': [40, 80, 120, 200],
    'model__l2_regularization': [0.0, 1e-3, 1e-2, 1e-1],
}
n_search = min(80_000, len(X_train))
Xs = X_train.sample(n_search, random_state=RANDOM_STATE); ys = y_train.loc[Xs.index]
search = RandomizedSearchCV(search_pipe, param_distributions=param_dist, n_iter=20,
                            scoring='neg_root_mean_squared_error', cv=3,
                            random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
search.fit(Xs, ys)
print('best params:', search.best_params_)
print('best CV RMSE:', -search.best_score_)
best_pipe = search.best_estimator_
best_pipe.fit(X_train, y_train)
best_metrics = models.regression_metrics(best_pipe, X_train, y_train, X_val, y_val, X_test, y_test)
import json
json.dump({'best_params': {k: (v if not isinstance(v,(int,float)) else float(v)) for k,v in search.best_params_.items()},
           'best_cv_rmse': float(-search.best_score_),
           'best_test_rmse': float(best_metrics.rmse_test),
           'best_test_r2': float(best_metrics.r2_test)},
          open(config.RES_DIR / 'tuning_summary.json','w'), indent=2)
best_metrics


## Courbe d'apprentissage

In [ ]:
from sklearn.model_selection import learning_curve
sizes = [0.1, 0.25, 0.5, 0.75, 1.0]
# learning_curve sur 80k pour rester rapide
Xlc = X_train.sample(80_000, random_state=RANDOM_STATE); ylc = y_train.loc[Xlc.index]
ts, tr_scores, va_scores = learning_curve(
    best_pipe, Xlc, ylc, train_sizes=sizes, cv=3,
    scoring='neg_root_mean_squared_error', n_jobs=-1, random_state=RANDOM_STATE
)
tr = -tr_scores.mean(axis=1); va = -va_scores.mean(axis=1)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ts, tr, 'o-', label='train', color='#1f6feb')
ax.plot(ts, va, 'o-', label='cv', color='#d62728')
ax.set_xlabel('# échantillons d\'entraînement'); ax.set_ylabel('RMSE')
ax.set_title('Courbe d\'apprentissage — HistGBRT'); ax.legend()
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'learning_curve.png', dpi=160); plt.show()


La RMSE val se stabilise vite. Ajouter des données aiderait peu, on est près de la borne du modèle.

## Importance + PDP

In [ ]:
Xv_s = X_val.sample(20_000, random_state=RANDOM_STATE)
imp = explain.perm_importance(best_pipe, Xv_s, y_val.loc[Xv_s.index])
imp.to_csv(config.RES_DIR / 'feature_importance.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(imp['feature'][::-1], imp['importance_mean'][::-1], color='#1f6feb')
ax.set_title('Importance par permutation (mesure : R²)')
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'perm_importance.png', dpi=160); plt.show()
imp


In [ ]:
explain.pdp_plot(best_pipe, X_train.sample(10_000, random_state=RANDOM_STATE),
                 ['heures_etude', 'assiduité_classe'],
                 out=config.FIG_DIR / 'pdp.png')
plt.show()


Permutation confirme : heures_etude domine, puis méthode_etude, qualité_sommeil, évaluation. Le reste est négligeable.

## Limites

Variables auto-déclarées (manipulables, cf. supplément). Granularité limitée. Pas de variable contextuelle (école, ressources).